# Fetch Heat Index Data Using CDC API

CDC only have heat index until 2022. Since the EDV data starts from 2017, we fetch the data from 2017 to 2022. To fetch the data, you need to email CDC and request an API token as of Sept 2025. You can also veiw the same data on https://ephtracking.cdc.gov/DataExplorer/.

The index is the maxinum daily Heat Index measured per county. The heat index was estimated using a modified version of the Rothfusz regression as implemented by the National Weather Service. Since it's on a county level, we aggregate it to a per region level using a  population-weighted approach.

In [1]:
# Fetch Raw Heat Index Data from CDC API
import os, time, random, json, csv, requests
from pathlib import Path
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

API_BASE = "https://ephtracking.cdc.gov/apigateway/api/v1"
MEASURE_ID = 358               # Heat Index
GEOGRAPHIC_TYPE_ID = 2         # County
STRATIFICATION_LEVEL_ID = 2    # County-level strata
TEMPORAL_TYPE_ID = 1           # Year
IS_SMOOTHED = 0
GET_FULL_CORE_HOLDER = 0
YEARS = list(range(2017, 2023))  # 2017..2022 inclusive

OUT_DIR = Path("../../data/heat_index").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

API_TOKEN = os.environ.get("CDC_TRACKING_API_TOKEN")

SESSION = requests.Session()
SESSION.headers.update({
    "Accept": "application/json",
    "User-Agent": "heat-index-harvester/1.0 (contact: you@example.com)"
})
retry = Retry(
    total=8,
    backoff_factor=2.0,  
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET", "POST"],
    respect_retry_after_header=True,
)
SESSION.mount("https://", HTTPAdapter(max_retries=retry))

def _params():
    return {"apiToken": API_TOKEN} if API_TOKEN else {}

def jitter_sleep(base_seconds: float):
    time.sleep(base_seconds + random.uniform(0, base_seconds))

def safe_request(fn, *args, **kwargs):
    try:
        r = fn(*args, **kwargs)
        r.raise_for_status()
        return r
    except requests.HTTPError as e:
        resp = getattr(e, "response", None)
        if resp is not None and resp.status_code == 429:
            ra = resp.headers.get("Retry-After")
            wait = float(ra) if ra else 30.0
            jitter_sleep(wait)
            r2 = fn(*args, **kwargs)
            r2.raise_for_status()
            return r2
        raise

def post_core_holder(geo_type_filter: str, geo_items_filter: str, temporal_items_filter: str):
    url = f"{API_BASE}/getCoreHolder/{MEASURE_ID}/{STRATIFICATION_LEVEL_ID}/{IS_SMOOTHED}/{GET_FULL_CORE_HOLDER}"
    body = {
        "geographicTypeIdFilter": geo_type_filter,     
        "geographicItemsFilter":  geo_items_filter,   
        "temporalTypeIdFilter":   str(TEMPORAL_TYPE_ID),
        "temporalItemsFilter":    temporal_items_filter, 
        "embedId": None
    }
    r = safe_request(SESSION.post, url, params=_params(), json=body, timeout=120)
    return r.json()

def flatten_rows(core_json):
    if not isinstance(core_json, dict):
        return []
    buckets = [
        "tableResult","healthImpactTableResult","climateChangeTableResult",
        "dailyEstimatesTableResult","heatEpisodesTableResult","pmTableResult",
        "cwsTableResult","sampleSizeTableResult","stateMetadataTableResult",
        "dailyTemperatureTableResult"
    ]
    for k in buckets:
        rows = core_json.get(k)
        if isinstance(rows, list):
            for row in rows:
                if isinstance(row, dict):
                    yield row

def get_autauga_2022():
    core = post_core_holder(str(GEOGRAPHIC_TYPE_ID), "1001", "2022")
    return list(flatten_rows(core)), core 

def fetch_all_counties_year(year: int):
    core = post_core_holder("ALL", "ALL", str(year))
    return list(flatten_rows(core))

def save_json(path: Path, data):
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

def save_jsonl(path: Path, rows):
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def save_csv(path: Path, rows):
    rows = list(rows)
    if not rows:
        path.write_text("", encoding="utf-8")
        return
    keys = sorted({k for r in rows for k in r.keys()})
    with path.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in rows:
            w.writerow(r)

In [ ]:
# Run this to make sure the API still works
autauga_rows, autauga_raw = get_autauga_2022()
save_json(OUT_DIR / "autauga_2022_heat_index_raw.json", autauga_raw)
save_csv(OUT_DIR / "autauga_2022_heat_index.csv", autauga_rows)
print(f"[demo] Saved Autauga 2022 -> {OUT_DIR}")

HTTPError: 405 Client Error:  for url: https://ephtracking.cdc.gov/apigateway/api/v1/getCoreHolder/358/2/0/0

In [ ]:
# Retrieve data. Will take a few minutes.
all_rows = []
for yr in YEARS:
    print(f"[pull] Year {yr} …")
    year_rows = fetch_all_counties_year(yr)
    print(f"       {len(year_rows)} rows")
    all_rows.extend(year_rows)
    jitter_sleep(5 if API_TOKEN else 30)

save_jsonl(OUT_DIR / "heat_index_all_counties_2017_2022.jsonl", all_rows)
save_csv(OUT_DIR / "heat_index_all_counties_2017_2022.csv", all_rows)
print(f"[done] Saved merged files -> {OUT_DIR}")


In [ ]:
# State Population average 
import pandas as pd

IN_MERGED = OUT_DIR / "heat_index_all_counties_2017_2022.csv"
POP_PATH = OUT_DIR / "US_population_2020.csv"
df = pd.read_csv(IN_MERGED, low_memory=False)

value_col = "dataValue" if "dataValue" in df.columns else ("DataValue" if "DataValue" in df.columns else None)
year_col  = "temporal"  if "temporal"  in df.columns else ("Year" if "Year" in df.columns else None)
state_col = "stateName" if "stateName" in df.columns else None

if state_col is None and "geographicItemsName" in df.columns:
    state_col = "stateName_synth"
    df[state_col] = df["geographicItemsName"].astype(str).str.split(",").str[-1].str.strip()

df = df[[c for c in [state_col, year_col, value_col] if c is not None]].dropna()
df[year_col] = df[year_col].astype(int)
df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
df = df.dropna(subset=[value_col])

df.head(3)

state_year = (
    df.groupby([state_col, year_col], as_index=False)[value_col]
      .mean()
      .rename(columns={state_col: "state", year_col: "year", value_col: "mean_heat_index"})
)

state_year_out = OUT_DIR / "state_year_heat_index.csv"
state_year.to_csv(state_year_out, index=False)
state_year.head(10)
pop_raw = pd.read_csv(POP_PATH)
total_row = pop_raw.loc[pop_raw.iloc[:,0].astype(str).str.strip().str.lower() == "total"].iloc[0]
state_pops = (
    total_row.drop(labels=total_row.index[0])   
              .rename_axis("state")
              .reset_index(name="pop_2020")
)
state_pops["pop_2020"] = (
    state_pops["pop_2020"].astype(str).str.replace(",", "", regex=False).astype(float)
)

state_pops["state"] = state_pops["state"].str.strip()
state_pops.head()
merged = state_year.merge(state_pops, on="state", how="left").dropna(subset=["pop_2020"])
nat = (
    merged.assign(weighted=lambda x: x["mean_heat_index"] * x["pop_2020"])
          .groupby("year", as_index=False)
          .apply(lambda g: pd.Series({
              "national_mean_heat_index_popweighted": g["weighted"].sum() / g["pop_2020"].sum()
          }))
)

nat_out = OUT_DIR / "national_popweighted_by_state_2020pop.csv"
nat.to_csv(nat_out, index=False)

In [ ]:
# CDC State Population average 

region_map = {
    "Connecticut": 1, "Maine": 1, "Massachusetts": 1, "New Hampshire": 1, "Rhode Island": 1, "Vermont": 1,
    "New Jersey": 2, "New York": 2, "Puerto Rico": 2, "US Virgin Islands": 2,
    "Delaware": 3, "District of Columbia": 3, "Maryland": 3, "Pennsylvania": 3, "Virginia": 3, "West Virginia": 3,
    "Alabama": 4, "Florida": 4, "Georgia": 4, "Kentucky": 4, "Mississippi": 4,
    "North Carolina": 4, "South Carolina": 4, "Tennessee": 4,
    "Illinois": 5, "Indiana": 5, "Michigan": 5, "Minnesota": 5, "Ohio": 5, "Wisconsin": 5,
    "Arkansas": 6, "Louisiana": 6, "New Mexico": 6, "Oklahoma": 6, "Texas": 6,
    "Iowa": 7, "Kansas": 7, "Missouri": 7, "Nebraska": 7,
    "Colorado": 8, "Montana": 8, "North Dakota": 8, "South Dakota": 8, "Utah": 8, "Wyoming": 8,
    "Arizona": 9, "California": 9, "Hawaii": 9, "Nevada": 9,
    "American Samoa": 9, "Commonwealth of the Northern Mariana Islands": 9,
    "Federated States of Micronesia": 9, "Guam": 9, "Marshall Islands": 9, "Republic of Palau": 9,
    "Alaska": 10, "Idaho": 10, "Oregon": 10, "Washington": 10,
}

sy = state_year.copy()
sy["region"] = sy["state"].map(region_map)
sy = sy.dropna(subset=["region"]).astype({"region": "int64"})

region_year = (
    sy.groupby(["region", "year"], as_index=False)["mean_heat_index"]
      .mean()
      .rename(columns={"mean_heat_index": "region_mean_heat_index"})
)

sy_w = sy.merge(state_pops[["state", "pop_2020"]], on="state", how="left").dropna(subset=["pop_2020"])
reg_pw = (
    sy_w.assign(w=lambda x: x["mean_heat_index"] * x["pop_2020"])
        .groupby(["region", "year"], as_index=False)
        .apply(lambda g: pd.Series({
            "region_mean_heat_index_popweighted_2020": g["w"].sum() / g["pop_2020"].sum()
        }))
)

region_year_out = OUT_DIR / "region_year_heat_index.csv"
reg_pw_out = OUT_DIR / "region_year_heat_index_popweighted_2020.csv"

region_year.to_csv(region_year_out, index=False)
reg_pw.to_csv(reg_pw_out, index=False)


